# Algèbre linéaire avec NumPy et SciPy

Ce notebook accompagne les diapos *4 - Inversion : Introduction*. Il reprend les exemples des diapos pour montrer comment :

1. construire la matrice $\mathbf{G}$ des exemples du cours ;
2. résoudre un système carré (élimination, forme ERL, factorisation LU) ;
3. vérifier l'indépendance des colonnes et l'existence de l'inverse ;
4. calculer le noyau $S_0(\mathbf{A})$, le rang et la dimension de $S_p(\mathbf{A})$ ;
5. estimer une matrice de covariance.

In [1]:
import numpy as np
import scipy.linalg as sl

np.set_printoptions(precision=4, suppress=True)

## 1. Construire $\mathbf{G}$

### Exemple 1 : ajuster une droite

Pour $T = a + bt$, chaque ligne de $\mathbf{G}$ vaut $[1,\ t_i]$.

In [2]:
t = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
G = np.column_stack((np.ones_like(t), t))
print(G)

[[1. 0.]
 [1. 1.]
 [1. 2.]
 [1. 3.]
 [1. 4.]]


### Exemple 2 : profilage sismique vertical

$G_{ij} = H(y_i - z_j)\,\Delta z$, où $z_j = (j+1)\Delta z$ est la profondeur de la base de la couche $j$ et $y_i$ la profondeur du géophone $i$. Avec $M = 10$ couches et $N = 5$ géophones, $M/N = 2$.

Le *broadcasting* (`y[:, None] - z[None, :]`) produit directement la matrice $N\times M$ des différences $y_i - z_j$. Le second argument de `np.heaviside` est la valeur retournée en 0, ici $H(0) = 1$.

In [3]:
M, N = 10, 5
dz = 1.0
dy = dz * M // N
z = (np.arange(M) + 1) * dz      # base des couches
y = (np.arange(N) + 1) * dy      # position des géophones

G_psv = dz * np.heaviside(y[:, None] - z[None, :], 1.0)
print(G_psv)

[[1. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 0. 0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1. 1. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1. 1. 1. 1. 0. 0.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]]


### Exemple 3 : tomographie (mur de briques)

Les briques sont numérotées colonne par colonne : la brique de la ligne `l` et de la colonne `c` porte l'indice `4*c + l`. Les mesures 0 à 3 traversent les colonnes et les mesures 4 à 7 traversent les lignes.

In [4]:
h = 1.0
G_tomo = np.zeros((8, 16))
for c in range(4):
    G_tomo[c, 4*c:4*c + 4] = h       # le long de la colonne c
for l in range(4):
    G_tomo[4 + l, l::4] = h          # le long de la ligne l
print(G_tomo.astype(int))

[[1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 1 1 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 1 1 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1]
 [1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0]
 [0 1 0 0 0 1 0 0 0 1 0 0 0 1 0 0]
 [0 0 1 0 0 0 1 0 0 0 1 0 0 0 1 0]
 [0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1]]


## 2. Résoudre un système carré

On reprend le système résolu par élimination de Gauss-Jordan dans les diapos :
\begin{align*}
x_1+2x_2+3x_3 &= 14 \\
x_1+2x_2+2x_3 &= 11 \\
x_1+3x_2+4x_3 &= 19
\end{align*}

`np.linalg.solve` effectue une factorisation LU suivie de deux substitutions. On ne calcule **jamais** `np.linalg.inv(A) @ b` pour résoudre un système : c'est plus coûteux et moins précis.

In [5]:
A = np.array([[1.0, 2.0, 3.0],
              [1.0, 2.0, 2.0],
              [1.0, 3.0, 4.0]])
b = np.array([14.0, 11.0, 19.0])

x = np.linalg.solve(A, b)
print(x)

[1. 2. 3.]


### Forme échelonnée réduite en lignes (ERL)

NumPy n'a pas de fonction pour la forme ERL : elle sert à raisonner, pas à calculer. La fonction ci-dessous applique l'élimination de Gauss-Jordan, avec choix du plus grand pivot pour limiter les erreurs d'arrondi. Elle retourne aussi les indices des colonnes pivot.

Pour un calcul exact, en fractions, on peut utiliser SymPy : `sympy.Matrix(A).rref()`.

In [6]:
def erl(A, tol=1e-10):
    # Forme échelonnée réduite en lignes et indices des colonnes pivot
    R = np.array(A, dtype=float)
    m, n = R.shape
    pivots = []
    i = 0
    for j in range(n):
        if i == m:
            break
        k = i + np.argmax(np.abs(R[i:, j]))    # plus grand pivot de la colonne
        if abs(R[k, j]) < tol:
            continue                            # pas de pivot : variable libre
        R[[i, k]] = R[[k, i]]                   # échange de lignes
        R[i] /= R[i, j]                         # pivot égal à 1
        for k in range(m):
            if k != i:
                R[k] -= R[k, j] * R[i]          # zéros dans le reste de la colonne
        pivots.append(j)
        i += 1
    R[np.abs(R) < tol] = 0.0
    return R, pivots

Ab = np.column_stack((A, b))      # matrice augmentée
R, piv = erl(Ab)
print(R)
print('colonnes pivot :', piv)

[[1. 0. 0. 1.]
 [0. 1. 0. 2.]
 [0. 0. 1. 3.]]
colonnes pivot : [0, 1, 2]


### Factorisation LU

`scipy.linalg.lu` retourne $\mathbf{P}$, $\mathbf{L}$ et $\mathbf{U}$ avec la convention $\mathbf{A} = \mathbf{PLU}$. Dans les diapos, la convention est $\mathbf{PA} = \mathbf{LU}$, ce qui correspond à $\mathbf{P}^T$ de SciPy. Ici les deux coïncident, parce que $\mathbf{P}$ ne fait qu'échanger deux lignes.

In [7]:
P, L, U = sl.lu(A)
print('P =\n', P)
print('L =\n', L)
print('U =\n', U)
print('PLU == A :', np.allclose(P @ L @ U, A))

P =
 [[1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]]
L =
 [[1. 0. 0.]
 [1. 1. 0.]
 [1. 0. 1.]]
U =
 [[ 1.  2.  3.]
 [ 0.  1.  1.]
 [ 0.  0. -1.]]
PLU == A : True


L'intérêt de la factorisation est qu'on la calcule une fois, en $O(n^3)$, puis qu'on la réutilise pour chaque nouveau membre de droite, en $O(n^2)$. `lu_factor` et `lu_solve` font exactement cela :

In [8]:
lu_piv = sl.lu_factor(A)
for b_k in ([14.0, 11.0, 19.0], [1.0, 0.0, 0.0], [6.0, 5.0, 8.0]):
    print(b_k, '->', sl.lu_solve(lu_piv, b_k))

[14.0, 11.0, 19.0] -> [1. 2. 3.]
[1.0, 0.0, 0.0] -> [ 2. -2.  1.]
[6.0, 5.0, 8.0] -> [1. 1. 1.]


## 3. Indépendance des colonnes et inverse

Pour la matrice $\mathbf{A} = [1\ 2\ 3;\ 4\ 5\ 6;\ 7\ 8\ 9]$, la forme ERL de $\mathbf{Ac} = \mathbf{0}$ montre que les colonnes sont dépendantes.

In [9]:
A9 = np.arange(1.0, 10.0).reshape(3, 3)
R9, piv9 = erl(A9)
print(R9)
print('colonnes pivot :', piv9)
print('rang :', np.linalg.matrix_rank(A9))

c = np.array([1.0, -2.0, 1.0])
print('A c =', A9 @ c)

[[ 1.  0. -1.]
 [ 0.  1.  2.]
 [ 0.  0.  0.]]
colonnes pivot : [0, 1]
rang : 2
A c = [0. 0. 0.]


**Attention aux arrondis.** En arithmétique flottante, le déterminant n'est pas exactement nul et `np.linalg.inv` peut retourner une « inverse » sans signaler d'erreur. Pour diagnostiquer une matrice singulière ou presque singulière, on utilise plutôt le rang (`matrix_rank`) ou le conditionnement (`cond`).

In [10]:
print('det  =', np.linalg.det(A9))
print('cond =', np.linalg.cond(A9))
try:
    print(np.linalg.inv(A9))
except np.linalg.LinAlgError as err:
    print('LinAlgError :', err)

det  = -9.51619735392994e-16
cond = 5.015946803014265e+16
[[-4.5036e+15  9.0072e+15 -4.5036e+15]
 [ 9.0072e+15 -1.8014e+16  9.0072e+15]
 [-4.5036e+15  9.0072e+15 -4.5036e+15]]


## 4. Noyau, rang et espace colonne

On reprend la matrice $3\times 4$ des diapos.

In [11]:
A = np.array([[3.0, 1.0,  9.0, 4.0],
              [2.0, 1.0,  7.0, 3.0],
              [5.0, 2.0, 16.0, 7.0]])
R, piv = erl(A)
print(R)
print('colonnes pivot :', piv)

[[1. 0. 2. 1.]
 [0. 1. 3. 1.]
 [0. 0. 0. 0.]]
colonnes pivot : [0, 1]


Les variables libres sont $x_3$ et $x_4$ (colonnes 2 et 3 en indexation Python). La forme ERL donne la base du noyau vue dans les diapos :
$$
\mathbf{v}_1 = [-2,\ -3,\ 1,\ 0]^T,\qquad \mathbf{v}_2 = [-1,\ -1,\ 0,\ 1]^T .
$$

`scipy.linalg.null_space` retourne une **autre** base du même sous-espace, orthonormée et calculée par SVD (voir les diapos suivantes du cours). On vérifie que $\mathbf{v}_1$ et $\mathbf{v}_2$ sont des combinaisons linéaires des colonnes de cette base : le résidu de la projection est nul.

In [12]:
V_erl = np.array([[-2.0, -3.0, 1.0, 0.0],
                  [-1.0, -1.0, 0.0, 1.0]]).T
N0 = sl.null_space(A)
print('base orthonormée du noyau :\n', N0)
print('A @ N0 ≈ 0 :', np.allclose(A @ N0, 0))

# projection de V_erl sur l'espace engendré par N0
resid = V_erl - N0 @ (N0.T @ V_erl)
print('résidu :', np.abs(resid).max())

base orthonormée du noyau :
 [[ 0.5923  0.0456]
 [ 0.6597  0.4602]
 [-0.0674 -0.4146]
 [-0.4576  0.7837]]
A @ N0 ≈ 0 : True
résidu : 1.687538997430238e-14


### Solution particulière et non-unicité

Avec $\mathbf{b} = [22\ 17\ 39]^T$, $\mathbf{p} = [1\ 2\ 1\ 2]^T$ est une solution, et $\mathbf{p}$ plus n'importe quel vecteur du noyau en est une aussi.

In [13]:
b = np.array([22.0, 17.0, 39.0])
p = np.array([1.0, 2.0, 1.0, 2.0])
print('A p =', A @ p)

rng = np.random.default_rng(0)
for _ in range(3):
    x = p + V_erl @ rng.normal(size=2)
    print(x, '-> A x =', A @ x)

A p = [22. 17. 39.]
[0.8806 1.7549 1.1257 1.8679] -> A x = [22. 17. 39.]
[-0.3857 -0.0262  1.6404  2.1049] -> A x = [22. 17. 39.]
[1.7097 3.2454 0.4643 2.3616] -> A x = [22. 17. 39.]


`np.linalg.solve` refuse ce système, qui n'est pas carré. `np.linalg.lstsq` retourne une solution parmi l'infinité possible : celle de norme minimale. Elle diffère de $\mathbf{p}$ d'un vecteur du noyau. Le choix d'une solution parmi d'autres est au cœur de l'inversion, et on y reviendra avec les moindres carrés et la SVD.

In [14]:
x_mn, res, rang, sv = np.linalg.lstsq(A, b)
print('x norme min. :', x_mn, ' norme =', np.linalg.norm(x_mn))
print('p            :', p, ' norme =', np.linalg.norm(p))
print('A x_mn =', A @ x_mn)
print('p - x_mn dans le noyau :', np.allclose(A @ (p - x_mn), 0))

x norme min. : [0.3529 0.4118 1.9412 0.7647]  norme = 2.155703788448081
p            : [1. 2. 1. 2.]  norme = 3.1622776601683795
A x_mn = [22. 17. 39.]
p - x_mn dans le noyau : True


### Théorème du rang

$\dim S_0(\mathbf{A}) + \dim S_p(\mathbf{A}) = n$, où $\dim S_p(\mathbf{A})$ est le rang. Vérifions-le pour les matrices de ce notebook, y compris celles des exemples du cours.

In [15]:
for nom, mat in [('A 3x4', A), ('A9', A9), ('droite', G),
                 ('PSV', G_psv), ('tomographie', G_tomo)]:
    r = np.linalg.matrix_rank(mat)
    d0 = sl.null_space(mat).shape[1]
    print(f'{nom:12s} {mat.shape}  rang = {r:2d}  dim S0 = {d0:2d}  '
          f'somme = {r + d0:2d}  n = {mat.shape[1]}')

A 3x4        (3, 4)  rang =  2  dim S0 =  2  somme =  4  n = 4
A9           (3, 3)  rang =  2  dim S0 =  1  somme =  3  n = 3
droite       (5, 2)  rang =  2  dim S0 =  0  somme =  2  n = 2
PSV          (5, 10)  rang =  5  dim S0 =  5  somme = 10  n = 10
tomographie  (8, 16)  rang =  7  dim S0 =  9  somme = 16  n = 16


Pour la tomographie, le noyau est de dimension 9 : avec 8 mesures et 16 briques, une infinité de modèles de lenteur expliquent parfaitement les données. Le rang vaut 7 et non 8, parce que la somme des temps le long des colonnes est égale à la somme des temps le long des lignes : les deux donnent la somme de toutes les briques multipliée par $h$. Une des 8 équations est donc redondante.

Pour le PSV, le noyau est de dimension 5 : seule la somme des lenteurs de chaque paire de couches entre deux géophones est déterminée.

## 5. Matrice de covariance

On simule $K$ réalisations de $N = 3$ données corrélées, regroupées dans $\mathbf{D}$ de taille $K\times N$, puis on compare la formule des diapos (division par $K$) avec `np.cov`.

`np.cov` suppose par défaut que **chaque ligne est une variable**. Avec $\mathbf{D}$ de taille $K\times N$, il faut donc passer `rowvar=False`. Par défaut, `np.cov` divise aussi par $K-1$ (estimateur non biaisé). `bias=True` donne la division par $K$.

In [16]:
C_vrai = np.array([[ 1.0, 0.8, -0.3],
                   [ 0.8, 1.0,  0.0],
                   [-0.3, 0.0,  0.5]])
K = 500
D = rng.multivariate_normal(mean=[0.0, 2.0, -1.0], cov=C_vrai, size=K)
print('D :', D.shape)

# formule des diapos
Dc = D - D.mean(axis=0)
C_est = Dc.T @ Dc / K
print('formule (1/K) :\n', C_est)

print('np.cov(D, rowvar=False, bias=True) identique :',
      np.allclose(np.cov(D, rowvar=False, bias=True), C_est))
print('np.cov(D, rowvar=False)  (1/(K-1)) :\n', np.cov(D, rowvar=False))
print('np.cov(D).shape sans rowvar=False :', np.cov(D).shape)

D : (500, 3)
formule (1/K) :
 [[ 0.9655  0.7734 -0.2659]
 [ 0.7734  0.9669  0.0029]
 [-0.2659  0.0029  0.4545]]
np.cov(D, rowvar=False, bias=True) identique : True
np.cov(D, rowvar=False)  (1/(K-1)) :
 [[ 0.9674  0.7749 -0.2665]
 [ 0.7749  0.9688  0.0029]
 [-0.2665  0.0029  0.4554]]
np.cov(D).shape sans rowvar=False : (500, 500)


## À essayer

- Modifiez `G_psv` pour $M/N = 1$ (un géophone par couche) : que deviennent le rang et le noyau ?
- Ajoutez au mur de briques deux mesures le long des diagonales : le rang augmente-t-il ? de combien ?
- Augmentez $K$ dans la dernière section et observez la convergence de la covariance estimée vers `C_vrai`.